In [73]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

In [74]:
df = pd.read_csv(r"E:\Pyhton code\Data Sets\Titanic\train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [75]:
df.shape

(891, 12)

In [76]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [77]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

In [78]:
#Checking the percentage of missing values in each column
missing_percentage = df.isnull().mean() * 100
missing_percentage.sort_values(ascending=False)

Cabin          77.104377
Age            19.865320
Embarked        0.224467
PassengerId     0.000000
Name            0.000000
Pclass          0.000000
Survived        0.000000
Sex             0.000000
Parch           0.000000
SibSp           0.000000
Fare            0.000000
Ticket          0.000000
dtype: float64

In [79]:
#Cabin contain 77% of missing value also it dosen't contain much information. But first letter of cabin indiactes deck. So we can extract that information and drop the cabin column.
df['Deck'] = df['Cabin'].str[0]

In [80]:
df.drop(columns=['Cabin'], inplace=True)

In [81]:
df.groupby("Deck")["Survived"].mean()*100

Deck
A    46.666667
B    74.468085
C    59.322034
D    75.757576
E    75.000000
F    61.538462
G    50.000000
T     0.000000
Name: Survived, dtype: float64

In [82]:
df["Deck"] = df["Deck"].fillna("Unknown")

In [83]:
df["Age"] = df.groupby(["Pclass", "Sex"])["Age"].transform(
    lambda x: x.fillna(x.median())
)

In [84]:
df.duplicated().sum()

np.int64(0)

In [85]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Embarked', 'Deck'],
      dtype='object')

In [86]:
df.drop(columns=['PassengerId'], inplace=True)

In [87]:
df['Title'] = df['Name'].str.split(',').str[1].str.split('.').str[0].str.strip()
df['Title'].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Col               2
Mlle              2
Major             2
Ms                1
Mme               1
Don               1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [88]:
rare_titles = ['Dr', 'Rev', 'Col', 'Major', 'Lady', 'Sir', 'Jonkheer', 'Don','Capt','the Countess']
df['Title'] = df['Title'].replace(rare_titles, 'Rare')
similar_titles_female = ['Ms', 'Mlle']
df['Title'] = df['Title'].replace(similar_titles_female, 'Miss')
similar_titles_male = ['Mme']
df['Title'] = df['Title'].replace(similar_titles_male, 'Mr')

In [89]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

In [90]:
df.drop(columns=['Name'], inplace=True)
df.drop(columns=['Ticket'], inplace=True)

In [91]:
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

In [92]:
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])


In [93]:
x = df.drop(columns=['Survived'])
y = df['Survived']

In [94]:
x.dtypes

Pclass          int64
Sex             int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked       object
Deck           object
Title          object
FamilySize      int64
dtype: object

In [95]:
df['Deck'].value_counts()

Deck
Unknown    687
C           59
B           47
D           33
E           32
A           15
F           13
G            4
T            1
Name: count, dtype: int64

In [96]:
x = pd.get_dummies(x, columns=['Deck', 'Title','Embarked'], drop_first=True)

In [97]:
#Getting the feature names after encoding (useful for later analysis)
feature_names = x.columns

In [98]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [99]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [100]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [101]:
y_pred = model.predict(x_test)

In [102]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.8268156424581006
Precision: 0.7945205479452054
Recall: 0.7837837837837838
F1 Score: 0.7891156462585034


In [103]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)

array([[90, 15],
       [16, 58]])

In [104]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.01, 0.1, 1, 10, 100]
}
grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv = 5,
    scoring= 'f1'
)

grid.fit(x_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best F1 Score:", grid.best_score_)

Best Parameters: {'C': 1}
Best F1 Score: 0.7663431712974662


In [105]:
LogisticRegression(penalty='l1', solver='liblinear')

,penalty,'l1'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [106]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(x_test)[:,1]
roc_auc_score(y_test, y_prob)

0.8854568854568855

In [107]:
#model interpretation

coef = pd.Series(model.coef_[0], index = feature_names)
coef.sort_values(key = abs, ascending=False).head(15)

Title_Mr     -1.483266
Title_Miss   -0.611612
Sex           0.605537
Pclass       -0.555617
SibSp        -0.365246
Age          -0.363703
Title_Rare   -0.357574
FamilySize   -0.326263
Deck_E        0.285498
Embarked_S   -0.226143
Fare          0.210716
Deck_D        0.174990
Parch        -0.150656
Deck_T       -0.149951
Deck_B        0.126695
dtype: float64

In [108]:
y_prob = model.predict_proba(x_test)[:,1]

threshold = 0.4
y_pred_custom = (y_prob > threshold).astype(int)

In [110]:
print("F1 Score:", f1_score(y_test, y_pred_custom))

F1 Score: 0.7741935483870968


In [111]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(x_train, y_train)

y_pred_rf = rf.predict(x_test)

from sklearn.metrics import accuracy_score
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

Random Forest Accuracy: 0.8212290502793296


🚢 Titanic Survival Prediction – End-to-End Machine Learning Project
📌 Problem Statement
Predict whether a passenger survived the Titanic disaster using passenger demographic and travel information.

This is a supervised binary classification problem.

📂 Dataset
Source: Kaggle – Titanic: Machine Learning from Disaster
Rows: 891
Target Variable: Survived (0 = No, 1 = Yes)

🧠 Project Objectives
Perform complete data cleaning and preprocessing,
Apply feature engineering techniques,
Train and evaluate Logistic Regression,
Tune hyperparameters using GridSearchCV,
Compare with Random Forest,
Build production-ready ML pipeline

🔍 Data Preprocessing Steps
Handled missing values:
Age → Group-based median imputation (Pclass + Sex)
Cabin → Extracted Deck, missing labeled as “Unknown”
Embarked → Mode imputation

Feature Engineering:
Extracted Title from Name
Grouped rare titles
Created FamilySize
Encoded categorical features
Dropped irrelevant columns (PassengerId, Ticket, Name)
Applied Standard Scaling (important for Logistic Regression)

🤖 Model Used

Logistic Regression with hyperparameter tuning (L2 regularization).

📊 Model Performance
Metric	Value
Accuracy	0.8268,
Precision	0.7945,
Recall	0.7838,
F1 Score	0.7891,
ROC-AUC	0.885

Random Forest Accuracy: 0.8212
Logistic Regression outperformed RF on this dataset.

📈 Key Insights
Gender (Female) strongly increases survival probability.
Higher passenger class improves survival chances.
Titles (Mrs, Miss, Master) positively correlated with survival.
Age and family size influence survival likelihood.

🏗 Production-Ready Pipeline
The entire preprocessing and model training workflow was converted into a single sklearn Pipeline to prevent data leakage and ensure reproducibility.

🛠 Tech Stack
Python,
Pandas,
NumPy,
Scikit-learn,
Matplotlib / Seaborn

```Pipeline Implementation```

In [ ]:
# import numpy as np
# import pandas as pd

# from sklearn.model_selection import train_test_split
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.impute import SimpleImputer
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# df = pd.read_csv(r"E:\Pyhton code\Data Sets\Titanic\train.csv")

In [ ]:
# #Feature Engineering
# def preprocess_features(df):

#     df = df.copy()

#     # Extract Title
#     df["Title"] = df["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)

#     # Normalize titles
#     df["Title"] = df["Title"].replace(["Mlle", "Ms"], "Miss")
#     df["Title"] = df["Title"].replace("Mme", "Mrs")

#     rare_titles = ["Dr", "Rev", "Col", "Major", "Don",
#                    "Lady", "Sir", "Capt", "the Countess", "Jonkheer"]
#     df["Title"] = df["Title"].replace(rare_titles, "Rare")

#     # Family Size
#     df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

#     # Deck from Cabin
#     df["Deck"] = df["Cabin"].str[0]
#     df["Deck"] = df["Deck"].fillna("Unknown")

#     # Drop unused columns
#     df = df.drop(["PassengerId", "Ticket", "Name", "Cabin"], axis=1)

#     return df

In [ ]:
# #Apply feature engineering
# df = preprocess_features(df)

# X = df.drop("Survived", axis=1)
# y = df["Survived"]

In [ ]:
# #Identify column types
# num_cols = X.select_dtypes(include=["int64", "float64"]).columns
# cat_cols = X.select_dtypes(include=["object"]).columns

In [ ]:
# #Build Preprocessing Pipeline
# num_pipeline = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("scaler", StandardScaler())
# ])

# cat_pipeline = Pipeline([
#     ("imputer", SimpleImputer(strategy="most_frequent")),
#     ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
# ])

# preprocessor = ColumnTransformer([
#     ("num", num_pipeline, num_cols),
#     ("cat", cat_pipeline, cat_cols)
# ])

In [ ]:
# #Final Pipeline
# final_pipeline = Pipeline([
#     ("preprocessing", preprocessor),
#     ("model", LogisticRegression(max_iter=1000, C=1))
# ])

In [ ]:
# #Train test split
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

# final_pipeline.fit(X_train, y_train)

In [ ]:
# #Evaluate the model
# y_pred = final_pipeline.predict(X_test)
# y_prob = final_pipeline.predict_proba(X_test)[:,1]

# print("Accuracy:", accuracy_score(y_test, y_pred))
# print("F1:", f1_score(y_test, y_pred))
# print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
# #Save the model
# import pickle

# with open("titanic_logistic_pipeline.pkl", "wb") as f:
#     pickle.dump(final_pipeline, f)